In [1]:
import openeo
from pathlib import Path
import pystac

In [2]:
from openeo.rest.stac_resource import StacResource
from openeo.internal.graph_building import PGNode

## Running FORCE level 2 with `run_cwl_to_stac`

This is the workflow that should be used in the end. Catalogs are not yet accessible from the result (as shown below).
The process only completes successfully when ran on `openeo-staging` (2026-03-26)

In [3]:
backend_url = "openeo-staging.dataspace.copernicus.eu"
connection = openeo.connect(backend_url).authenticate_oidc()

Authenticated using refresh token.


In [4]:
import yaml
from yaml import CLoader

In [5]:
with open("../test/force-l2-params.yml") as fp:
    example_params = yaml.load(fp, CLoader)

In [6]:
example_params

{'aoi': '{ "type": "Feature", "geometry": { "type": "Polygon", "coordinates": [[[10.6,44.5],[10.6,44.8],[11.0,44.8],[11.0,44.5],[10.6,44.5]]] }, "properties": { "name": "Parma" }, "id": "08" }',
 'buffer_nodata': False,
 'cirrus_buffer': 30,
 'cloud_buffer': 300,
 'cloud_threshold': 0.1,
 'dem': 'Copernicus_30m',
 'do_adjacency': False,
 'do_aod': True,
 'do_atmo': True,
 'do_brdf': True,
 'do_multi_scattering': True,
 'do_topo': True,
 'erase_clouds': False,
 'impulse_noise': True,
 'input': ['s3://eodata/Sentinel-2/MSI/L1C/2024/11/13/S2A_MSIL1C_20241113T101251_N0511_R022_T32TPQ_20241113T121135.SAFE'],
 'max_cloud_cover_frame': 90,
 'max_cloud_cover_tile': 90,
 'nproc': 3,
 'nthread': 4,
 'origin_lon': -25,
 'origin_lat': 60,
 'output_aod': False,
 'output_dst': False,
 'output_format': 'GTiff',
 'output_hot': False,
 'output_ovv': False,
 'output_vzn': False,
 'output_wvp': False,
 'parallel_reads': False,
 'process_start_delay': 3,
 'projection': 'GLANCE7',
 'res_merge': 'IMPROPHE',

In [7]:
cwl_url = "https://raw.githubusercontent.com/bcdev/apex-force-openeo/refs/heads/main/cwl/force-l2.cwl"
#context = dict(
#    input=["s3://EODATA/Sentinel-2/MSI/L1C/2024/11/13/S2A_MSIL1C_20241113T101251_N0511_R022_T32TPQ_20241113T121135.SAFE"],
#    #aoi='{ "type": "Feature", "geometry": { "type": "Polygon", "coordinates": [[[10.6,44.5],[10.6,44.8],[11.0,44.8],[11.0,44.5],[10.6,44.5]]] }, "properties": { "name": "Parma" }, "id": "08" }',
#)
context = example_params

stac_resource = StacResource(
    graph=PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl_url": cwl_url,
            "context": context,
        }
    ),
    connection=connection,
)
force_job = stac_resource.create_job(title="FORCE level 2")
force_job.start_and_wait()

0:00:00 Job 'j-26032714482046d9a89ee02e7488f19b': send 'start'
0:00:14 Job 'j-26032714482046d9a89ee02e7488f19b': created (progress 0%)
0:00:19 Job 'j-26032714482046d9a89ee02e7488f19b': created (progress 0%)
0:00:26 Job 'j-26032714482046d9a89ee02e7488f19b': created (progress 0%)
0:00:34 Job 'j-26032714482046d9a89ee02e7488f19b': created (progress 0%)
0:00:44 Job 'j-26032714482046d9a89ee02e7488f19b': created (progress 0%)
0:00:56 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 8.3%)
0:01:12 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 10.4%)
0:01:31 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 12.9%)
0:01:55 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 15.9%)
0:02:25 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 19.3%)
0:03:03 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 23.2%)
0:03:50 Job 'j-26032714482046d9a89ee02e7488f19b': running (progress 27.5%)
0:04:48 Job 'j-26032714482046d9a89ee02e7488f19b': run

<BatchJob job_id='j-26032714482046d9a89ee02e7488f19b'>

In [8]:
force_results = force_job.get_results()
force_results

<JobResults for job 'j-26032714482046d9a89ee02e7488f19b'>

In [9]:
catalog_link = next(iter(l for l in force_results.get_metadata()["links"] if l["href"].endswith("catalogue.json")))
catalog_url = f"https://{backend_url}/{catalog_link['href']}"
print("Assets:")
print("\t", force_results.get_metadata()["assets"])
print("Catalog link:")
print("\t", catalog_link)

Assets:
	 {'europe/X0031_Y0029/20241113_LEVEL2_SEN2A_BOA.tif': {'eo:bands': [{'name': 'B2'}, {'name': 'B3'}, {'name': 'B4'}, {'name': 'B5'}, {'name': 'B6'}, {'name': 'B7'}, {'name': 'B8'}, {'name': 'B8A'}, {'name': 'B11'}, {'name': 'B12'}], 'file:size': 38640954, 'href': 'https://s3.stag.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-staging-waw4-1/batch_jobs/j-26032714482046d9a89ee02e7488f19b/europe/X0031_Y0029/20241113_LEVEL2_SEN2A_BOA.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=8cdee574e7ac44408ec040f98a9b8674%2F20260327%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260327T145610Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXN0YWdpbmctd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnN0YWcud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsia

The catalog indicated in the job results does not exist:

In [10]:
catalog_url

'https://openeo-staging.dataspace.copernicus.eu//batch_jobs/j-26032714482046d9a89ee02e7488f19b/catalogue.json'

In [11]:
!curl -I {catalog_url}

HTTP/1.1 404 Not Found
Content-Length: 153
Content-Type: text/html
Date: Fri, 27 Mar 2026 14:56:11 GMT



We can however access the catalog (and from it, the item) through the logs:

In [12]:
force_logs = force_job.logs()
filtered_logs = (l.message for l in force_logs if "catalog.json" in l.message or "catalogue.json" in l.message)
print(f"\n{'-'*80}\n".join(filtered_logs))

                "location": "file:///calrissian/output-data/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json",
--------------------------------------------------------------------------------
                "basename": "catalogue.json",
--------------------------------------------------------------------------------
                "path": "/calrissian/output-data/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json"
--------------------------------------------------------------------------------
result 'l2-ard/catalogue.json': v.generate_public_url()='https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json' v.generate_presigned_url()='https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=2b362f5724b14d019bbddaeee22f303e%2F20260327%2FWAW3-1%

In [13]:
log_with_item_url = next(iter(l.message for l in force_logs if "copy_asset(" in l.message and ("catalogue.json" in l.message or "catalog.json" in l.message)))
log_with_item_url

'URL: copy_asset(https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json)'

In [14]:
print(log_with_item_url)
catalog_url = log_with_item_url.removeprefix("URL: copy_asset(").removesuffix(")")
catalog_url

URL: copy_asset(https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json)


'https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/catalogue.json'

In [15]:
item = next(iter((pystac.Catalog.from_file(catalog_url).get_items())))
item_url = next(iter(item.get_links("self"))).href
item_url

'https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/cube-20260327T1454-l2-ard.json'

### Running FORCE TSA

In [16]:
packed_cwl = "/tmp/force-tsa-packed.cwl"

In [17]:
!cwltool --pack ../../apex-force-openeo/cwl/force-tsa-workflow.cwl ../../apex-force-openeo/cwl/force-tsa.cwl ../../apex-force-openeo/cwl/force-tsa-parameter-schema.yaml > {packed_cwl}

INFO /home/hannes/micromamba/envs/testing/bin/cwltool 3.1.20260315121657
INFO Resolved '../../apex-force-openeo/cwl/force-tsa-workflow.cwl' to 'file:///home/hannes/code/apex-force-openeo/cwl/force-tsa-workflow.cwl'


In [18]:
# process parameters
context = dict(
    item_url=item_url,
    DATE_RANGE_START="2024-01-01",
    DATE_RANGE_END="2024-12-31",
)

stac_resource = StacResource(
    graph=PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl": Path(packed_cwl).read_text(),
            "context": context,
        },
    ),
    connection=connection,
)
tsa_job = stac_resource.create_job(title="FORCE TSA")
tsa_job.start_and_wait()

<BatchJob job_id='j-2603271456154d95b8f00c491abeeff9'>

In [29]:
tsa_results = tsa_job.get_results()
tsa_results

<JobResults for job 'j-2603271456154d95b8f00c491abeeff9'>

In [33]:
tsa_logs = tsa_job.logs()
filtered_logs = (l.message for l in tsa_logs if "catalog.json" in l.message or "catalogue.json" in l.message)
print(f"\n{'-'*80}\n".join(filtered_logs))

Job spec: {
 "process_graph": {
  "runcwltostac1": {
   "process_id": "run_cwl_to_stac",
   "arguments": {
    "context": {
     "item_url": "https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-26032714482046d9a8-cal-cwl-39cb91cb/l2-ard/cube-20260327T1454-l2-ard.json",
     "DATE_RANGE_START": "2024-01-01",
     "DATE_RANGE_END": "2024-12-31"
    },
    "cwl": "{\n    \"$graph\": [\n        {\n            \"class\": \"Workflow\",\n            \"requirements\": [\n                {\n                    \"types\": [\n                        {\n                            \"type\": \"enum\",\n                            \"name\": \"#force-tsa-parameter-schema.yaml/STM_type\",\n                            \"label\": \"Spectral Temporal Metrics to include\",\n                            \"symbols\": [\n                                \"#force-tsa-parameter-schema.yaml/STM_type/MIN\",\n                                \"#force-tsa-parameter-schema.yaml/STM_t

In [35]:
log_with_catalog_url = next(iter(l.message for l in tsa_logs if "copy_asset(" in l.message and ("catalogue.json" in l.message or "catalog.json" in l.message)))
print(log_with_catalog_url)
catalog_url = log_with_catalog_url.removeprefix("URL: copy_asset(").removesuffix(")")
catalog_url

URL: copy_asset(https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2603271456154d95b8-cal-cwl-8ca774cc/catalog.json)


'https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2603271456154d95b8-cal-cwl-8ca774cc/catalog.json'